# Exploratory Data Analysis — Paysim Dataset

Download theA quick overview of the dataset: structure, distributions, class imbalance, and correlations.

## Read Data from S3 bucket

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

In [ ]:
# TODO: Load the raw data from S3 - URI can be copied from the S3 console
raw_s3_uri = ""

In [ ]:
print(f"Loading data from S3 URI: {raw_s3_uri}")
df = pd.read_csv(raw_s3_uri)

You may get error for missing packages like fsspec and s3fs. You can install them using uv : 
```
uv add fsspec s3fs
```

## Shape & Schema

In [ ]:
print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')
df.dtypes.to_frame('dtype')

In [ ]:
df.head(5)

## Missing Values & Duplicates

In [ ]:
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'None')
print(f'\nDuplicate rows: {df.duplicated().sum():,}')

## Numerical Summary

In [ ]:
df.describe()

## Target Variable — `isFraud`

In [ ]:
fraud_counts = df['isFraud'].value_counts()
print('Class distribution:')
print(fraud_counts)
print(f'\nFraud rate: {df.isFraud.mean()*100:.4f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fraud_counts.plot.pie(ax=axes[0], labels=['Legit', 'Fraud'], autopct='%1.2f%%',
                      colors=['#4CAF50', '#F44336'], startangle=90)
axes[0].set_title('Class Balance')
axes[0].set_ylabel('')

sns.countplot(x='isFraud', data=df, ax=axes[1], palette=['#4CAF50', '#F44336'])
axes[1].set_title('Transaction Count by Fraud Label')
axes[1].set_xticklabels(['Legit (0)', 'Fraud (1)'])

plt.tight_layout()
plt.show()

## Transaction Type Breakdown

In [ ]:
df['type'].value_counts()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['type'].value_counts().plot.bar(ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Transaction Type Distribution')
axes[0].set_ylabel('Count')

fraud_by_type = df.groupby('type')['isFraud'].mean().sort_values(ascending=False)
fraud_by_type.plot.bar(ax=axes[1], color='salmon', edgecolor='black')
axes[1].set_title('Fraud Rate by Transaction Type')
axes[1].set_ylabel('Fraud Rate')
axes[1].set_ylim(0, 1)

for i, v in enumerate(fraud_by_type):
    axes[1].text(i, v + 0.02, f'{v:.2%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## Amount Distribution (Overall & by Fraud)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['amount'].hist(bins=60, ax=axes[0], color='steelblue', edgecolor='black', log=True)
axes[0].set_title('Transaction Amount Distribution (log-scale)')
axes[0].set_xlabel('Amount')
axes[0].set_ylabel('Frequency (log)')

for label, color in zip([0, 1], ['#4CAF50', '#F44336']):
    subset = df[df['isFraud'] == label]['amount']
    subset.hist(bins=60, ax=axes[1], alpha=0.6, color=color, edgecolor='black',
                label='Fraud' if label else 'Legit', log=True)
axes[1].set_title('Amount Distribution by Class (log-scale)')
axes[1].set_xlabel('Amount')
axes[1].legend()

plt.tight_layout()
plt.show()

## Balance Features — Fraud vs Legit

In [ ]:
balance_cols = ['oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col in zip(axes.ravel(), balance_cols):
    for label, color in zip([0, 1], ['#4CAF50', '#F44336']):
        df[df['isFraud'] == label][col].hist(
            bins=50, ax=ax, alpha=0.6, color=color,
            label='Fraud' if label else 'Legit', log=True)
    ax.set_title(col)
    ax.legend()

plt.suptitle('Balance Distributions by Fraud Label', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Correlation Heatmap

In [ ]:
numeric_cols = ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
                'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']

corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## `isFlaggedFraud` Analysis

In [ ]:
df[df['isFlaggedFraud'] == 1]

In [ ]:
print('isFlaggedFraud value counts:')
print(df['isFlaggedFraud'].value_counts())

flagged_fraud = df[(df['isFlaggedFraud'] == 1) & (df['isFraud'] == 1)].shape[0]
total_fraud = df[df['isFraud'] == 1].shape[0]
print(f'\nFlagged fraud caught: {flagged_fraud}/{total_fraud} ({flagged_fraud/total_fraud*100:.2f}%)')

## Key Observations

1. **Severe class imbalance** — fraud is a tiny fraction of all transactions.

2. **Transaction type matters** — `TRANSFER` and `CASH_OUT` carry the highest fraud rates; `CASH_IN`, `PAYMENT`, `DEBIT` are mostly legit.

3. **Amount alone is not enough** — fraud and legit amounts overlap heavily in the lower range.

4. **`isFlaggedFraud` is a weak rule** — it catches only a small fraction of actual frauds.

5. **Caveat for training** — the balance columns look like strong fraud signal here, but that's
   a PaySim simulator artifact (e.g. `newbalanceOrig` often drops straight to `0` for fraud,
   regardless of `amount`), not real-world signal. If a trained model reports `validation-auc`
   near `1.0`, this is why — treat it as leakage to investigate, not a good result.